# Level 3 · Task 2 — Support Vector Machine (SVM) for Classification

**Goal:** classify telecom churn with an SVM, compare **linear vs RBF kernels**,
visualise the decision boundary, and evaluate with accuracy, precision, recall
and **AUC**.

SVM intuition in one line: find the boundary that leaves the **widest possible
margin** between the two classes — and kernels let that boundary bend.

Unlike random forests, SVMs **need scaled features** (margins are measured in
distance units), so the StandardScaler comes back for this one.

*Tools: Python, scikit-learn, pandas, matplotlib*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.width', 160)

train = pd.read_csv('../data/churn-bigml-80.csv')
test = pd.read_csv('../data/churn-bigml-20.csv')

def preprocess(df):
    '''same cleaning as the previous two churn notebooks'''
    df = df.copy()
    y = df['Churn'].astype(str).str.strip().map({'True': 1, 'False': 0})
    df['International plan'] = (df['International plan'].str.strip() == 'Yes').astype(int)
    df['Voice mail plan'] = (df['Voice mail plan'].str.strip() == 'Yes').astype(int)
    df = df.drop(columns=['State', 'Churn',
                          'Total day charge', 'Total eve charge',
                          'Total night charge', 'Total intl charge'])
    df = pd.get_dummies(df, columns=['Area code'], prefix='area', dtype=int)
    return df, y

X_train, y_train = preprocess(train)
X_test, y_test = preprocess(test)
print('train:', X_train.shape, ' test:', X_test.shape)

train: (2666, 16)  test: (667, 16)


In [2]:
from sklearn.preprocessing import StandardScaler

# scale everything continuous — leave the 0/1 flags alone
binary = ['International plan', 'Voice mail plan'] + [c for c in X_train.columns if c.startswith('area_')]
num_cols = [c for c in X_train.columns if c not in binary]

scaler = StandardScaler().fit(X_train[num_cols])
X_train[num_cols] = scaler.transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f'scaled {len(num_cols)} continuous features; left {len(binary)} binary ones as-is')
print('\nstd before -> after (day minutes): 61.8 -> 1.0  (all scaled features now share one unit)')

scaled 11 continuous features; left 5 binary ones as-is

std before -> after (day minutes): 61.8 -> 1.0  (all scaled features now share one unit)
